# Agentic AI Systems — Worklet 13
# Research Grant Management System (apniLeap AURA)
## Lab 4 · Memory and Retrieval — State vs Stored (Context Design)

**Phase A · Foundations** &nbsp;|&nbsp; **Goal:** Separate short-term run state from long-term stored knowledge

---

**Description**

Separates short-term session state (the active proposal draft, temporary calculations)
from long-term stored knowledge (approved literature corpus, historical grant outcomes,
and institutional policy decisions).

**Key concepts demonstrated**

- Strict separation of working memory (session state) and long-term persistent store.
- Structured MemoryRecord with topic, fact, date, author, and source attribution.
- Scoped retrieval (no full memory dumping; topic-scoped and tag-filtered lookup).
- TF-IDF / keyword similarity retrieval over the academic literature corpus.
- Conflict detection: flags conflicts with existing policy rather than silently overwriting.
- Permission-gated write access: only authorized roles ('dean', 'research_office') may write.
- LangChain Retriever adapter demonstration.

**Assignments included**

- Assignment 1: Isolated topic retrieval (proving zero leakage between 'overhead_rate' and 'ethics').
- Assignment 2: Staleness / retention check (warning when retrieved decisions exceed age threshold).

---

*Runs standalone on Python 3.10+ (Jupyter or Google Colab): no API key, no network, standard library only. Source script: [`scripts/lab04_memory_and_retrieval.py`](./scripts/lab04_memory_and_retrieval.py).*

### Cell 1 · Shared institutional seed data

`seed_data.py` is written to disk so the notebook runs standalone. It is the Python mirror of the course project's `backend/src/seed.js`: funding calls, literature corpus, policies, personnel rate card, users and the sanctioned-project portfolio. *(Collapse — shared by all labs.)*

In [1]:
%%writefile seed_data.py
"""
AURA Seed Data & Corpus -- Research Grant Management System
Corresponds to backend/src/seed.js, faculty.js, and projects.js
"""

INSTITUTION = {
    "id": "inst-mist",
    "name": "Meridian Institute of Science and Technology",
    "short": "MIST",
    "type": "university",
    "city": "Pune",
    "country": "IN",
    "researchOffice": "Research & Development Office",
    "signatory": "Dean, Research & Development",
}

USERS = [
    {
        "id": "usr-krishnan",
        "name": "Dr. Uma Mudengudi",
        "email": "uma.mudengudi@mist.edu.in",
        "role": "dean",
        "title": "Dean, Research & Development",
        "department": "Office of Dean R&D",
        "permissions": ["run:create", "run:read:all", "review:approve", "package:view", "audit:read", "version:rollback", "metrics:read"],
    },
    {
        "id": "usr-suneeta-v-budihal",
        "name": "Dr. Suneeta V Budihal",
        "email": "suneeta.v.budihal@mist.edu.in",
        "role": "pi",
        "title": "Associate Professor",
        "department": "SoECE",
        "permissions": ["run:create", "run:read:own", "draft:edit", "package:view"],
    },
    {
        "id": "usr-satyadhyan-chickerur",
        "name": "Dr. Satyadhyan Chickerur",
        "email": "satyadhyan.chickerur@mist.edu.in",
        "role": "pi",
        "title": "Professor & Head",
        "department": "CSE",
        "permissions": ["run:create", "run:read:own", "draft:edit", "package:view"],
    },
    {
        "id": "usr-research-office",
        "name": "Research & Development Office",
        "email": "grants.cell@mist.edu.in",
        "role": "research_office",
        "title": "Grants Cell & Compliance",
        "department": "Office of Dean R&D",
        "permissions": ["run:create", "run:read:all", "review:approve", "package:view", "audit:read", "version:rollback"],
    },
]

RATE_CARD = {
    "institutionId": "inst-mist",
    "overheadPct": 15,
    "contingencyPct": 5,
    "salaryEscalationPct": 5,
    "equipmentMaintenancePct": 4,
    "effectiveFrom": "2026-04-01",
    "note": "Institutional default rates. Agency-published rates for a call always take precedence.",
}

PERSONNEL = [
    {"id": "per-uma", "name": "Dr. Uma Mudengudi", "role": "PI", "title": "Dean (R&D) & Professor", "department": "SoECE", "monthlyCost": 0, "defaultEffortPct": 10, "careerStage": "professor"},
    {"id": "per-suneeta", "name": "Dr. Suneeta V Budihal", "role": "PI", "title": "Faculty Member", "department": "SoECE", "monthlyCost": 0, "defaultEffortPct": 10, "careerStage": "associate"},
    {"id": "per-satyadhyan", "name": "Dr. Satyadhyan Chickerur", "role": "PI", "title": "Faculty Member", "department": "CSE", "monthlyCost": 0, "defaultEffortPct": 10, "careerStage": "associate"},
    {"id": "per-ra", "name": "Research Associate (to be appointed)", "role": "RA", "title": "Research Associate", "department": "Research Office", "monthlyCost": 48000, "defaultEffortPct": 100, "careerStage": "n/a"},
    {"id": "per-phd", "name": "Doctoral Scholar (to be appointed)", "role": "PhD", "title": "Doctoral Scholar", "department": "Research Office", "monthlyCost": 36000, "defaultEffortPct": 100, "careerStage": "n/a"},
]

POLICIES = [
    {
        "id": "pol-overhead",
        "title": "Institutional overhead and costing policy",
        "tags": ["budget", "overhead"],
        "text": "Overheads are charged at the rate published by the funding agency for the call. Where the agency publishes no rate, the institutional default of 15% of total direct costs applies. Overhead may not fund permanent faculty salaries.",
    },
    {
        "id": "pol-ethics",
        "title": "Human subjects research ethics policy",
        "tags": ["compliance", "ethics", "health"],
        "text": "Any proposal involving human participants, identifiable health data or clinical validation requires Institutional Ethics Committee approval before the award is activated. A conditional certificate may be attached at submission and must be converted to full approval before funds are released.",
    },
    {
        "id": "pol-data",
        "title": "Research data protection policy (DPDP-aligned)",
        "tags": ["compliance", "data", "privacy"],
        "text": "Proposals collecting personal data must include a Data Management Plan covering lawful basis, de-identification, storage location, retention period and a named data steward. Cross-border transfer of health data requires a documented risk assessment.",
    },
    {
        "id": "pol-ai",
        "title": "Disclosure of AI-assisted writing",
        "tags": ["compliance", "ai", "authorship"],
        "text": "AI-assisted drafting must be disclosed in the proposal cover declaration and in the institutional submission record. AI-generated text is a draft until a named faculty member reviews and accepts it; AI tools may not be listed as authors or investigators.",
    },
    {
        "id": "pol-coi",
        "title": "Conflict of interest declaration",
        "tags": ["compliance", "coi"],
        "text": "Every investigator must declare financial interests, advisory roles and institutional relationships that could reasonably be perceived to influence the research. Declarations are collected by the Research Office and retained for the audit period.",
    },
    {
        "id": "pol-signoff",
        "title": "Authorised signatory policy",
        "tags": ["compliance", "approval"],
        "text": "Only the Dean, Research & Development, or a delegated Research Office signatory, may authorise submission to a funding agency. The authorising officer must record the decision, the version reviewed, and any conditions attached to approval.",
    },
]

COMPLIANCE_RULES = [
    {"id": "rule-sections", "kind": "sections", "appliesTo": "all", "severity": "blocking", "requirement": "Every section required by the call is present and non-empty.", "evidence": "Call required-section list vs draft section index."},
    {"id": "rule-pages", "kind": "pageLimit", "appliesTo": "all", "severity": "blocking", "requirement": "The narrative stays within the page limit published for the call.", "evidence": "Word count converted at 500 words per page."},
    {"id": "rule-budget-ceiling", "kind": "budgetCeiling", "appliesTo": "all", "severity": "blocking", "requirement": "The requested total does not exceed the agency ceiling for this call.", "evidence": "Budget total vs call award maximum."},
    {"id": "rule-equipment-cap", "kind": "equipmentCap", "appliesTo": "all", "severity": "blocking", "requirement": "Equipment does not exceed the percentage ceiling in the call guidelines.", "evidence": "Equipment subtotal / total requested."},
    {"id": "rule-salary-cap", "kind": "salaryCap", "appliesTo": "all", "severity": "blocking", "requirement": "Manpower costs stay within the call salary ceiling.", "evidence": "Personnel subtotal / total requested."},
    {"id": "rule-overhead", "kind": "overheadRate", "appliesTo": "all", "severity": "blocking", "requirement": "Overhead is charged at exactly the rate published for the call.", "evidence": "Overhead amount vs rate x direct costs."},
    {"id": "rule-ethics", "kind": "document", "docKey": "ethics", "appliesTo": "all", "severity": "blocking", "requirement": "Ethics clearance is attached where the call or the research involves human participants.", "evidence": "Document register vs call requirement."},
    {"id": "rule-coi", "kind": "document", "docKey": "coi", "appliesTo": "all", "severity": "blocking", "requirement": "Conflict-of-interest declarations are attached for all investigators.", "evidence": "Document register vs call requirement."},
    {"id": "rule-dmp", "kind": "document", "docKey": "dmp", "appliesTo": "all", "severity": "blocking", "requirement": "A Data Management Plan is attached where required.", "evidence": "Document register vs call requirement."},
    {"id": "rule-disclosure", "kind": "aiDisclosure", "appliesTo": "all", "severity": "blocking", "requirement": "AI-assisted drafting is disclosed in the submission declaration.", "evidence": "Institutional AI disclosure policy vs package declaration."},
]

DOCUMENT_REGISTER = [
    {"key": "ethics", "label": "Institutional Ethics Committee clearance", "status": "conditional", "note": "Conditional certificate available; full approval pending protocol review."},
    {"key": "coi", "label": "Conflict-of-interest declarations (all investigators)", "status": "attached", "note": "Signed by all investigators."},
    {"key": "dmp", "label": "Data Management Plan", "status": "attached", "note": "Deposited in the institutional DMP registry."},
    {"key": "budget-justification", "label": "Budget justification", "status": "attached", "note": "Generated by the Budget Planning Agent."},
    {"key": "csr-1", "label": "CSR-1 registration and company CSR policy", "status": "pending", "note": "Company secretarial certificate requested."},
    {"key": "consortium-agreement", "label": "Consortium agreement and partner letters", "status": "pending", "note": "Letters of intent collected; agreement under legal review."},
]

CALLS = [
    {
        "id": "call-serb-crg",
        "agency": "SERB",
        "agencyFull": "Science and Engineering Research Board, DST",
        "program": "Core Research Grant (CRG)",
        "title": "SERB Core Research Grant 2026 -- Individual Investigator",
        "fundingType": "government",
        "domains": ["ai-ml", "energy", "materials", "health", "climate"],
        "tags": ["artificial intelligence", "machine learning", "federated learning", "sustainable energy", "materials science", "medical imaging"],
        "deadline": "2026-12-15",
        "budgetMin": 3000000,
        "budgetMax": 7500000,
        "durationMonths": 36,
        "pageLimit": 25,
        "verified": True,
        "status": "open",
        "eligibility": {"careerStages": ["assistant", "associate", "professor"], "institutionTypes": ["university", "institute"], "consortiumRequired": False, "minPartners": 0, "coFundingRequired": False},
        "budgetRules": {"overheadPct": 15, "salaryCapPct": 45, "equipmentCapPct": 40, "travelCapPct": 10, "contingencyPct": 5},
        "guidelines": [
            {"id": "call-serb-crg#g1", "section": "submission", "text": "Proposals must be routed through the institutional research office; individual submissions are not accepted."},
            {"id": "call-serb-crg#g2", "section": "budget", "text": "The total request may not exceed INR 75,00,000 for a duration of up to 36 months."},
            {"id": "call-serb-crg#g3", "section": "budget", "text": "Equipment may not exceed 40% of the total requested budget; requests above this ceiling require a written justification from the Head of Institution."},
            {"id": "call-serb-crg#g4", "section": "budget", "text": "Overheads are payable at 15% of total direct costs."},
            {"id": "call-serb-crg#g5", "section": "documents", "text": "A Data Management Plan and a conflict-of-interest declaration are mandatory at submission."},
            {"id": "call-serb-crg#g6", "section": "format", "text": "The narrative may not exceed 25 pages excluding references and the budget annexure."},
        ],
    },
    {
        "id": "call-icmr-adhoc",
        "agency": "ICMR",
        "agencyFull": "Indian Council of Medical Research",
        "program": "Ad-hoc Research Project",
        "title": "ICMR Ad-hoc Research Project 2026 -- Diagnostics & Public Health",
        "fundingType": "government",
        "domains": ["health", "ai-ml", "public-health"],
        "tags": ["diabetic retinopathy", "medical imaging", "rural health", "screening", "federated learning", "digital health", "primary care"],
        "deadline": "2026-12-05",
        "budgetMin": 2000000,
        "budgetMax": 6000000,
        "durationMonths": 36,
        "pageLimit": 25,
        "verified": True,
        "status": "open",
        "eligibility": {"careerStages": ["assistant", "associate", "professor"], "institutionTypes": ["university", "institute", "hospital"], "consortiumRequired": False, "minPartners": 0, "coFundingRequired": False},
        "budgetRules": {"overheadPct": 15, "salaryCapPct": 50, "equipmentCapPct": 30, "travelCapPct": 10, "contingencyPct": 5},
        "guidelines": [
            {"id": "call-icmr-adhoc#g1", "section": "eligibility", "text": "Applicants must hold a regular position in a recognised institution or hospital and demonstrate access to the study population."},
            {"id": "call-icmr-adhoc#g2", "section": "budget", "text": "The total request must fall between INR 20,00,000 and INR 60,00,000; equipment is capped at 30%."},
            {"id": "call-icmr-adhoc#g3", "section": "ethics", "text": "Institutional Ethics Committee approval is mandatory for any study involving human participants or their data."},
            {"id": "call-icmr-adhoc#g4", "section": "data", "text": "A Data Management Plan describing de-identification and retention is required; identifiable health data may not leave the institution without a documented risk assessment."},
            {"id": "call-icmr-adhoc#g5", "section": "deliverables", "text": "The proposal must state measurable clinical or public-health outcomes for the funding period."},
        ],
    },
    {
        "id": "call-tata-csr",
        "agency": "Tata Trusts",
        "agencyFull": "Tata Trusts -- Corporate Social Responsibility",
        "program": "CSR Research Partnership",
        "title": "Tata Trusts CSR Research Partnership 2026 -- Health & Education",
        "fundingType": "industry",
        "domains": ["health", "education", "public-health"],
        "tags": ["community health", "digital health", "education technology", "rural outreach", "screening"],
        "deadline": "2026-11-15",
        "budgetMin": 2500000,
        "budgetMax": 10000000,
        "durationMonths": 24,
        "pageLimit": 20,
        "verified": True,
        "status": "open",
        "eligibility": {"careerStages": ["assistant", "associate", "professor"], "institutionTypes": ["university", "institute", "hospital", "ngo"], "consortiumRequired": False, "minPartners": 0, "coFundingRequired": False},
        "budgetRules": {"overheadPct": 8, "salaryCapPct": 55, "equipmentCapPct": 25, "travelCapPct": 10, "contingencyPct": 3},
        "guidelines": [
            {"id": "call-tata-csr#g1", "section": "compliance", "text": "The activity must fall within Schedule VII of the Companies Act 2013 and the implementing partner must hold a valid CSR-1 registration."},
            {"id": "call-tata-csr#g2", "section": "budget", "text": "Grants range from INR 25,00,00,000 to INR 1,00,00,000 over 24 months; overheads are capped at 8%."},
            {"id": "call-tata-csr#g3", "section": "reporting", "text": "Quarterly outcome reporting against agreed indicators is a condition of release of the second tranche."},
        ],
    },
    {
        "id": "call-eu-horizon",
        "agency": "EU Horizon Europe",
        "agencyFull": "European Commission -- Horizon Europe",
        "program": "Cluster 4: Digital, Industry and Space",
        "title": "Horizon Europe 2027 -- Trustworthy AI for Health Systems (RIA)",
        "fundingType": "international",
        "domains": ["ai-ml", "health", "digital"],
        "tags": ["trustworthy ai", "federated learning", "health systems", "digital health", "data governance", "open science"],
        "deadline": "2027-01-20",
        "budgetMin": 18000000,
        "budgetMax": 45000000,
        "durationMonths": 36,
        "pageLimit": 45,
        "verified": True,
        "status": "open",
        "eligibility": {"careerStages": ["assistant", "associate", "professor"], "institutionTypes": ["university", "institute", "hospital"], "consortiumRequired": True, "minPartners": 3, "coFundingRequired": False},
        "budgetRules": {"overheadPct": 25, "salaryCapPct": 40, "equipmentCapPct": 30, "travelCapPct": 15, "contingencyPct": 0},
        "guidelines": [
            {"id": "call-eu-horizon#g1", "section": "eligibility", "text": "Proposals require a consortium of at least three independent legal entities from three different countries."},
            {"id": "call-eu-horizon#g2", "section": "budget", "text": "Personnel costs may not exceed 40% and equipment 30% of the total eligible cost; a 25% flat rate covers indirect costs."},
            {"id": "call-eu-horizon#g3", "section": "open-science", "text": "Open science practices, FAIR data management and a gender and inclusion plan are evaluated under the impact criterion."},
        ],
    },
    {
        "id": "call-serb-supra-expired",
        "agency": "SERB",
        "agencyFull": "Science and Engineering Research Board, DST",
        "program": "SUPRA",
        "title": "SERB SUPRA 2026 -- Closed",
        "fundingType": "government",
        "domains": ["ai-ml", "health"],
        "tags": ["artificial intelligence", "medical imaging"],
        "deadline": "2026-06-30",
        "budgetMin": 1000000,
        "budgetMax": 3000000,
        "durationMonths": 24,
        "status": "closed",
        "verified": True,
        "eligibility": {"careerStages": ["assistant", "associate", "professor"], "institutionTypes": ["university"], "consortiumRequired": False, "minPartners": 0},
        "budgetRules": {"overheadPct": 15, "salaryCapPct": 45, "equipmentCapPct": 40, "travelCapPct": 10, "contingencyPct": 5},
        "guidelines": [{"id": "call-serb-supra-expired#g1", "section": "status", "text": "This call closed on 30 June 2026 and is retained for reference only."}],
    },
    {
        "id": "call-unverified-sample",
        "agency": "Unknown Aggregator",
        "agencyFull": "Third-party opportunity aggregator",
        "program": "Sample Aggregator Call",
        "title": "Unverified Grant Opportunity 2027",
        "fundingType": "government",
        "domains": ["engineering"],
        "tags": ["engineering"],
        "deadline": "2027-02-01",
        "budgetMin": 800000,
        "budgetMax": 2500000,
        "durationMonths": 24,
        "verified": False,
        "status": "unverified",
        "eligibility": {"careerStages": ["assistant"], "institutionTypes": ["college"], "consortiumRequired": False, "minPartners": 0},
        "budgetRules": {"overheadPct": 10, "salaryCapPct": 40, "equipmentCapPct": 35, "travelCapPct": 12, "contingencyPct": 5},
        "guidelines": [{"id": "call-unverified#g1", "section": "status", "text": "Listing could not be confirmed against an official agency notification."}],
    },
]

LITERATURE = [
    {
        "id": "lit-001",
        "title": "Federated learning for diabetic retinopathy screening across rural clinics",
        "authors": ["S. Banerjee", "R. Iyer", "M. Kulkarni"],
        "year": 2024,
        "venue": "Journal of Medical AI Research",
        "tags": ["federated learning", "diabetic retinopathy", "rural health", "medical imaging", "screening"],
        "abstract": "A four-site federated study reports comparable sensitivity to centralised training while keeping 1.2 million fundus images inside participating clinics.",
        "citations": 61,
        "doi": "10.5555/aura.demo.001",
    },
    {
        "id": "lit-002",
        "title": "Deep learning detection of referable diabetic retinopathy in primary care",
        "authors": ["L. Fernandes", "A. Gupta"],
        "year": 2023,
        "venue": "Ophthalmology Informatics",
        "tags": ["diabetic retinopathy", "medical imaging", "primary care", "screening", "deep learning"],
        "abstract": "A prospective primary-care evaluation of 8,400 patients reports 92% sensitivity and 88% specificity for referable retinopathy.",
        "citations": 148,
        "doi": "10.5555/aura.demo.002",
    },
    {
        "id": "lit-003",
        "title": "Privacy-preserving federated learning in healthcare: a systematic review",
        "authors": ["H. Park", "J. Okafor", "N. Singh"],
        "year": 2024,
        "venue": "Health Data Science Review",
        "tags": ["federated learning", "privacy", "health systems", "data governance"],
        "abstract": "Reviews 74 studies and finds that differential privacy and secure aggregation are rarely combined in clinical deployments.",
        "citations": 96,
        "doi": "10.5555/aura.demo.003",
    },
    {
        "id": "lit-004",
        "title": "Non-mydriatic fundus photography in resource-limited settings",
        "authors": ["P. Adjei", "T. Nguyen"],
        "year": 2022,
        "venue": "Global Eye Health",
        "tags": ["rural health", "screening", "medical imaging", "primary care"],
        "abstract": "Image quality is the dominant failure mode for non-mydriatic cameras operated by trained community health workers.",
        "citations": 57,
        "doi": "10.5555/aura.demo.004",
    },
    {
        "id": "lit-005",
        "title": "Teleophthalmology screening pathways in low-resource districts",
        "authors": ["K. Rao", "D. Bhattacharya"],
        "year": 2023,
        "venue": "Indian Journal of Public Health",
        "tags": ["screening", "rural health", "health systems", "telemedicine"],
        "abstract": "Hub-and-spoke teleophthalmology reduced time-to-diagnosis from 26 to 9 days across six districts.",
        "citations": 73,
        "doi": "10.5555/aura.demo.005",
    },
]

HISTORY = [
    {
        "id": "hist-retina-2024",
        "title": "AI-assisted retinal screening for diabetic retinopathy in Pune district",
        "agency": "ICMR",
        "year": 2024,
        "outcome": "not-funded",
        "amount": 5600000,
        "tags": ["diabetic retinopathy", "screening", "rural health", "medical imaging"],
        "abstract": "Proposed a three-site pilot of automated retinal grading with referral support for 12,000 patients.",
        "lessons": "Reviewers questioned the missing ethics clearance and the absence of a site-level data-governance plan. Both are now prerequisites.",
    },
    {
        "id": "hist-fl-testbed-2023",
        "title": "Federated learning testbed for multi-hospital medical imaging",
        "agency": "SERB",
        "year": 2023,
        "outcome": "funded",
        "amount": 6200000,
        "tags": ["federated learning", "medical imaging", "privacy", "testbed"],
        "abstract": "Built a three-hospital federated learning testbed with secure aggregation and site-level audit logging.",
        "lessons": "Delivered the testbed; clinical validation was out of scope and remains the open gap.",
    },
]


Writing seed_data.py


### Cell 2 · Imports and fixtures

In [2]:
import math
import re
from dataclasses import dataclass, field
from datetime import datetime, date, timedelta
from typing import List, Dict, Any, Optional
from seed_data import LITERATURE, HISTORY, POLICIES

### Cell 3 · `MemoryRecord` (class)

In [3]:
# ==============================================================================
# Structured Memory Models & Record Types
# ==============================================================================

@dataclass
class MemoryRecord:
    topic: str
    fact: str
    date: str
    source: str = "memory"
    author_role: str = "research_office"
    metadata: Dict[str, Any] = field(default_factory=dict)

### Cell 4 · `ProjectMemoryStore` (class)

In [4]:
class ProjectMemoryStore:
    """
    Long-term persistent memory store with role-gated write permissions
    and scoped retrieval.
    """
    def __init__(self):
        self.records: List[MemoryRecord] = []
        self.allowed_writers = {"dean", "research_office"}

    def write(self, record: MemoryRecord, user_role: str):
        """Permission-gated write path."""
        if user_role not in self.allowed_writers:
            raise PermissionError(f"Role '{user_role}' is not permitted to write to institutional memory.")
        self.records.append(record)

    def retrieve(self, topic: str) -> List[MemoryRecord]:
        """Scoped retrieval: returns only records matching the topic."""
        return [r for r in self.records if r.topic.lower() == topic.lower()]


# ==============================================================================
# TF-IDF Cosine Retrieval over Academic Corpus (from backend/src/core.js)
# ==============================================================================

STOPWORDS = {
    'a', 'an', 'and', 'are', 'as', 'at', 'be', 'by', 'for', 'from', 'has', 'have',
    'in', 'into', 'is', 'it', 'its', 'of', 'on', 'or', 'that', 'the', 'their', 'this',
    'to', 'was', 'were', 'will', 'with', 'within', 'we', 'our', 'can', 'not', 'but'
}

### Cell 5 · `tokenize` (function)

In [5]:
def tokenize(text: str) -> List[str]:
    clean = re.sub(r'[^a-zA-Z0-9\s-]', ' ', str(text).lower())
    return [w for w in clean.split() if len(w) > 2 and w not in STOPWORDS]

### Cell 6 · `CorpusRetriever` (class)

In [6]:
class CorpusRetriever:
    """
    Retrieves relevant prior literature and proposal history using TF-IDF cosine similarity.
    Matches the algorithm in backend/src/core.js.
    """
    def __init__(self, docs: List[Dict[str, Any]], text_field_fn):
        self.docs = docs
        self.tfs = [self._term_freq(tokenize(text_field_fn(d))) for d in docs]
        self.df = {}
        for tf in self.tfs:
            for term in tf:
                self.df[term] = self.df.get(term, 0) + 1
        n = len(docs) or 1
        self.idf = {term: math.log(1 + n / (1 + freq)) for term, freq in self.df.items()}
        self.vectors = []
        for tf in self.tfs:
            vec = {t: f * self.idf.get(t, 1.0) for t, f in tf.items()}
            self.vectors.append(vec)

    @staticmethod
    def _term_freq(tokens: List[str]) -> Dict[str, int]:
        tf = {}
        for t in tokens:
            tf[t] = tf.get(t, 0) + 1
        return tf

    @staticmethod
    def _cosine(v1: Dict[str, float], v2: Dict[str, float]) -> float:
        dot = sum(v1[k] * v2[k] for k in v1 if k in v2)
        norm1 = math.sqrt(sum(v * v for v in v1.values()))
        norm2 = math.sqrt(sum(v * v for v in v2.values()))
        if not norm1 or not norm2:
            return 0.0
        return dot / (norm1 * norm2)

    def retrieve_similar(self, query: str, k: int = 3) -> List[Dict[str, Any]]:
        q_tokens = tokenize(query)
        q_tf = self._term_freq(q_tokens)
        q_vec = {t: f * self.idf.get(t, math.log(2)) for t, f in q_tf.items()}

        scored = []
        for i, doc in enumerate(self.docs):
            score = self._cosine(q_vec, self.vectors[i])
            if score > 0:
                scored.append({"doc": doc, "similarity": round(score, 4)})

        scored.sort(key=lambda x: x["similarity"], reverse=True)
        return scored[:k]

### Cell 7 · `LangChainRetrieverAdapter` (class)

In [7]:
# ==============================================================================
# LangChain Retriever Adapter Pattern
# ==============================================================================

class LangChainRetrieverAdapter:
    """
    Adapter demonstrating standard LangChain BaseRetriever contract:
    get_relevant_documents(query) -> List[Document].
    """
    def __init__(self, corpus_retriever: CorpusRetriever):
        self.retriever = corpus_retriever

    def get_relevant_documents(self, query: str):
        results = self.retriever.retrieve_similar(query)
        # Convert to LangChain-style Document format
        return [
            {
                "page_content": f"{r['doc']['title']} - {r['doc'].get('abstract', '')}",
                "metadata": {"id": r["doc"]["id"], "score": r["similarity"]},
            }
            for r in results
        ]

### Cell 8 · `ProjectAssistant` (class)

In [8]:
# ==============================================================================
# Project Assistant (Working Memory vs Stored Memory)
# ==============================================================================

class ProjectAssistant:
    def __init__(self, memory_store: ProjectMemoryStore, lit_retriever: CorpusRetriever):
        self.memory_store = memory_store
        self.lit_retriever = lit_retriever
        self.session_state = {}  # Working state (current active draft)

    def answer_topic(self, topic: str, staleness_days: Optional[int] = None) -> str:
        """Answers from scoped long-term memory with explicit source citation."""
        hits = self.memory_store.retrieve(topic)
        if not hits:
            return f"[session] No stored institutional decision on '{topic}' yet."

        latest = hits[-1]
        warning = ""
        if staleness_days:
            rec_date = datetime.strptime(latest.date, "%Y-%m-%d").date()
            age_days = (date.today() - rec_date).days
            if age_days > staleness_days:
                warning = f"[STALENESS WARNING: decision is {age_days} days old and may be out of date] "

        return f"{warning}[memory, dated {latest.date}] Per institutional decision on '{latest.topic}': {latest.fact}"

    def receive_new_statement(self, topic: str, new_fact: str, user_role: str) -> Dict[str, Any]:
        """
        Conflict detection: compares against latest decision before writing.
        """
        hits = self.memory_store.retrieve(topic)
        if hits and hits[-1].fact.lower() != new_fact.lower():
            return {
                "conflict": True,
                "stored": hits[-1].fact,
                "stored_date": hits[-1].date,
                "new": new_fact,
                "message": (
                    f"CONFLICT DETECTED: Proposed fact '{new_fact}' conflicts with "
                    f"existing decision from {hits[-1].date} ('{hits[-1].fact}'). "
                    f"Requires Dean approval to overwrite."
                ),
            }

        # Safe to write (still permission checked)
        record = MemoryRecord(
            topic=topic,
            fact=new_fact,
            date=str(date.today()),
            author_role=user_role,
        )
        self.memory_store.write(record, user_role)
        return {"conflict": False, "written": True, "message": f"Recorded new fact for '{topic}'."}

### Cell 9 · `run_assignment_1_isolated_topics` (function)

In [9]:
# ==============================================================================
# Graded Assignments
# ==============================================================================

def run_assignment_1_isolated_topics(assistant: ProjectAssistant):
    """
    Assignment 1 -- Two Isolated Topics:
    Demonstrate that querying 'overhead_rate' returns only overhead facts and never
    leaks facts from 'ethics_policy', and vice versa.
    """
    print("\n--- [Assignment 1] Isolated Topic Scoping Test ---")
    assistant.receive_new_statement(
        "overhead_rate", "Institutional overhead is set to 15% of direct costs", user_role="dean"
    )
    assistant.receive_new_statement(
        "ethics_policy", "Clinical trials must have Institutional Ethics Committee clearance", user_role="dean"
    )

    ans_overhead = assistant.answer_topic("overhead_rate")
    ans_ethics = assistant.answer_topic("ethics_policy")

    print(f"Query 'overhead_rate': {ans_overhead}")
    print(f"Query 'ethics_policy': {ans_ethics}")

    assert "ethics" not in ans_overhead.lower(), "Leakage detected: ethics mentioned in overhead answer!"
    assert "overhead" not in ans_ethics.lower(), "Leakage detected: overhead mentioned in ethics answer!"
    print("Verification passed: Strict topic boundary enforced with zero leakage.")

### Cell 10 · `run_assignment_2_staleness_policy` (function)

In [10]:
def run_assignment_2_staleness_policy(assistant: ProjectAssistant):
    """
    Assignment 2 -- Staleness / Retention Policy:
    Add a retention_days threshold. If retrieved record exceeds threshold,
    warn the user before answering.
    """
    print("\n--- [Assignment 2] Staleness & Retention Policy Test ---")
    # Fresh record (today)
    assistant.receive_new_statement("salary_ceiling", "JRF salary capped at 37,000 INR/month", user_role="dean")
    # Artificially old record (2 years ago)
    old_date = str(date.today() - timedelta(days=730))
    assistant.memory_store.records.append(
        MemoryRecord(topic="legacy_hardware_grant", fact="Servers require local physical rack", date=old_date, author_role="dean")
    )

    fresh_ans = assistant.answer_topic("salary_ceiling", staleness_days=365)
    stale_ans = assistant.answer_topic("legacy_hardware_grant", staleness_days=365)

    print(f"Fresh Record (under 365 days): {fresh_ans}")
    print(f"Stale Record (over 365 days): {stale_ans}")
    assert "STALENESS WARNING" in stale_ans, "Staleness policy failed to trigger warning!"
    print("Verification passed: Staleness policy warned user about outdated record.")

### Cell 11 · Run the lab

The lab demo and both graded assignments, with the console captured for the checks below.

In [11]:
# Console capture: everything the lab prints is shown below AND kept in memory,
# so the acceptance checks in the last cell can assert on the real run.
import io
import sys


class ConsoleTee(io.StringIO):
    """Mirror stdout into a buffer while the lab runs."""

    def __enter__(self):
        self._stdout = sys.stdout
        sys.stdout = self
        return self

    def __exit__(self, *exc):
        sys.stdout = self._stdout
        return False

    def write(self, text):
        self._stdout.write(text)
        return super().write(text)


console = ConsoleTee()

with console:
    print("======================================================================")
    print("Lab 4: Memory and Retrieval -- State vs Stored")
    print("======================================================================")

    # 1. Initialize persistent memory store with seed policy
    mem_store = ProjectMemoryStore()
    mem_store.write(
        MemoryRecord(
            topic="data_residency",
            fact="Patient health data must remain inside participating hospital firewalls",
            date="2026-07-03",
            author_role="dean",
        ),
        user_role="dean",
    )

    # 2. Build TF-IDF retriever for literature corpus
    lit_retriever = CorpusRetriever(
        docs=LITERATURE,
        text_field_fn=lambda d: f"{d['title']} {d['abstract']} {' '.join(d['tags'])}"
    )

    assistant = ProjectAssistant(mem_store, lit_retriever)

    # 3. Retrieve policy fact
    print("\n=== Initial Policy Retrieval ===")
    print(assistant.answer_topic("data_residency"))

    # 4. Conflict detection test
    print("\n--- Conflict Detection: Faculty member proposes cloud migration ---")
    conflict_result = assistant.receive_new_statement(
        topic="data_residency",
        new_fact="Migrate all raw patient fundus images to public US cloud bucket",
        user_role="pi",
    )
    print(conflict_result["message"])

    # 5. Permission violation test
    print("\n--- Permission Gate: Unauthorized role attempting to write ---")
    try:
        mem_store.write(
            MemoryRecord(topic="grant_decision", fact="Approve immediately", date="2026-11-15"),
            user_role="student",
        )
    except PermissionError as e:
        print(f"Blocked as expected: {e}")

    # 6. TF-IDF Literature retrieval demo
    print("\n=== Literature Corpus Cosine Retrieval ===")
    query = "federated learning diabetic retinopathy screening"
    top_lit = lit_retriever.retrieve_similar(query, k=2)
    for hit in top_lit:
        print(f"[{hit['doc']['id']}] Score: {hit['similarity']} | {hit['doc']['title']}")

    # 7. Run Assignments 1 and 2
    run_assignment_1_isolated_topics(assistant)
    run_assignment_2_staleness_policy(assistant)

Lab 4: Memory and Retrieval -- State vs Stored

=== Initial Policy Retrieval ===
[memory, dated 2026-07-03] Per institutional decision on 'data_residency': Patient health data must remain inside participating hospital firewalls

--- Conflict Detection: Faculty member proposes cloud migration ---
CONFLICT DETECTED: Proposed fact 'Migrate all raw patient fundus images to public US cloud bucket' conflicts with existing decision from 2026-07-03 ('Patient health data must remain inside participating hospital firewalls'). Requires Dean approval to overwrite.

--- Permission Gate: Unauthorized role attempting to write ---
Blocked as expected: Role 'student' is not permitted to write to institutional memory.

=== Literature Corpus Cosine Retrieval ===
[lit-001] Score: 0.6186 | Federated learning for diabetic retinopathy screening across rural clinics
[lit-002] Score: 0.4653 | Deep learning detection of referable diabetic retinopathy in primary care

--- [Assignment 1] Isolated Topic Scoping Te

### Cell 12 · Acceptance checks and deliverables

Each check asserts on the trace of the run above. The trace and the check report are exported to `lab4_artifacts/` (committed under `Course-Project/evidence/`).

In [12]:
import json
import re
from pathlib import Path

LAB = 4
CHECKS = [('Policy retrieval by tag returns stored reference', 'Initial Policy Retrieval', 1),
    ('Literature retrieved by TF-IDF cosine similarity', 'Literature Corpus Cosine Retrieval', 1),
    ('Topic scopes are isolated with zero leakage', 'Verification passed: Strict topic boundary enforced with zero leakage', 1),
    ('Stale records raise a warning instead of being trusted', 'Verification passed: Staleness policy warned user', 1)]

trace = console.getvalue()
universal = [
    ("Run produced console output", r"\S", 1),
    ("Graded Assignment 1 executed", r"\[Assignment 1\]", 1),
    ("Graded Assignment 2 executed", r"\[Assignment 2\]", 1),
]
results = []
for label, pattern, minimum in universal + CHECKS:
    found = len(re.findall(pattern, trace, flags=re.MULTILINE))
    results.append({"check": label, "pattern": pattern, "expected_min": minimum,
                    "found": found, "passed": found >= minimum})
no_errors = "Traceback (most recent call last)" not in trace
results.append({"check": "No unhandled exception in the run", "pattern": "Traceback",
                "expected_min": 0, "found": 0 if no_errors else 1, "passed": no_errors})

for i, r in enumerate(results, 1):
    print(f"[{'PASS' if r['passed'] else 'FAIL'}] {i:>2}. {r['check']}")
passed = sum(r["passed"] for r in results)
print(f"\nLab {LAB}: {passed}/{len(results)} acceptance checks passed")

outdir = Path(f"lab{LAB}_artifacts")
outdir.mkdir(exist_ok=True)
(outdir / f"lab{LAB}_console_trace.txt").write_text(trace, encoding="utf-8")
(outdir / f"lab{LAB}_acceptance_checks.json").write_text(
    json.dumps({"lab": LAB, "passed": passed, "total": len(results), "checks": results}, indent=2),
    encoding="utf-8")
print("Artifacts written to", outdir.as_posix())

assert passed == len(results), "acceptance checks failed"

[PASS]  1. Run produced console output
[PASS]  2. Graded Assignment 1 executed
[PASS]  3. Graded Assignment 2 executed
[PASS]  4. Policy retrieval by tag returns stored reference
[PASS]  5. Literature retrieved by TF-IDF cosine similarity
[PASS]  6. Topic scopes are isolated with zero leakage
[PASS]  7. Stale records raise a warning instead of being trusted
[PASS]  8. No unhandled exception in the run

Lab 4: 8/8 acceptance checks passed
Artifacts written to lab4_artifacts
